In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader('./backend/db_pipeline/data/encykorea_cleaned6.csv', encoding='utf-8')

docs = loader.load()

In [ ]:
print(f"문서의 수: {len(docs)}")

In [ ]:
docs[0].metadata

In [ ]:
print(docs[0].page_content[:5000])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    # 청크 크기를 매우 작게 설정합니다. 예시를 위한 설정입니다.
    chunk_size=5000,
    # 청크 간의 중복되는 문자 수를 설정합니다.
    chunk_overlap=300,
    # 문자열 길이를 계산하는 함수를 지정합니다.
    length_function=len,
    # 구분자로 정규식을 사용할지 여부를 설정합니다.
    is_separator_regex=False,
    # (옵션)한국어 문장 구분용 구분자 설정
    separators=[
        "\n\n",  # 문단
        "\n",    # 줄바꿈
        r"(?<=[.?!])\s+",  # 영어 문장
        r"(?<=[다요죠음함임니다])[\.\?!]?\s+",  # 한국어 종결 표현
        " ",      # 단어
        "",       # 최후의 수단: 글자 단위
    ]
)

In [ ]:
docs_with_splitter = text_splitter.split_documents(docs)
print(f"문서의 수: {len(docs_with_splitter)}")

In [ ]:
docs_with_splitter[0].metadata

In [ ]:
print(docs_with_splitter[0].page_content)

In [ ]:
import pandas as pd

df = pd.read_csv("./backend/db_pipeline/data/encykorea_cleaned6_202512040954.csv")

df

In [ ]:
import pandas as pd

df = pd.read_csv("./backend/db_pipeline/data/encykorea_cleaned6.csv")

df

In [79]:
import pandas as pd

df = pd.read_csv("./backend/db_pipeline/data/encykorea_cleaned6.csv")

# # 전체 category 컬럼만 보기
# print(df["category"])

# 고유 category 목록만 보기
print(df["category"].dropna().unique())

# # 혹은 정렬된 목록
# print(sorted(df["category"].dropna().unique()))

['물품' '문헌' '제도' '사건' '개념' '인물' '지명' '작품' '유적' '의례·행사' '단체' '의복' '정책']


In [84]:
import pandas as pd
import re

df = pd.read_csv("./backend/db_pipeline/data/encykorea_cleaned6.csv")

keywords = ["왕", "역할"]
pattern = "|".join(map(re.escape, keywords))  # OR 패턴

mask = df["contents"].str.contains(pattern, na=False)

# 중복 제거된 category 목록
cats = df.loc[mask, "category"].dropna().unique()
print("category 목록:", cats)

# # 정렬된 목록이 필요하면
# print("정렬된 category 목록:", sorted(cats))

# 필터된 행 개수
print("행 개수:", mask.sum())

category 목록: ['문헌' '제도' '물품' '사건' '개념' '인물' '지명' '유적' '의례·행사' '작품' '단체' '의복' '정책']
행 개수: 4182


In [92]:
import pandas as pd
import re

pd.set_option("display.max_colwidth", None)

df = pd.read_csv("./backend/db_pipeline/data/encykorea_cleaned6.csv")

keywords = ["왕"]
pattern = "|".join(map(re.escape, keywords))
mask = df["contents"].str.contains(pattern, na=False)

target_cat = "의복"

filtered = df.loc[mask & (df["category"] == target_cat), ["category", "contents"]]

# 매 실행마다 랜덤 순서
shuffled = filtered.sample(frac=1).reset_index(drop=True)

for idx, row in shuffled.iterrows():
    print(f"[{idx}] category: {row['category']}\n{str(row['contents']).replace('\\n', '\n')}")
    print("-" * 40)

print("선택 카테고리 행 개수:", len(filtered))


[0] category: 의복
중국에서는 한대 이래로 관인의 복식 하나로서 관품에 따라 옥 · 금 · 서 · 은 · 유석으로 만든 관대를 착용하였다. 당대의 경우 1∼3품은 금옥대, 4·5품은 금대, 6·7품은 은대, 8·9품은 유석대를 착용케 하는 등 관대제를 크게 정비하였다. 원대는 제 · 공 · 의위 복별과 관품별로, 옥 · 서 · 금 · 여지금 등으로 만든 관대를 착용하였다. 명나라에서는 1370년(태조 3) 문무관의 상복 상정과 함께 1품은 옥대, 2품은 화서대, 3품은 금삽화대, 4품은 소금대, 5품은 은삽화대, 6·7품은 소은대, 8·9품은 오각대의 품대식을 정하였다. 1393년(태조 2)에 다시 조 · 제 · 공복의 품대(상복의 품대와 거의 비슷)를 상정하면서 크게 정비되어 변동없이 명 말까지 계승되었다. 우리나라의 경우, 삼국 · 통일신라 · 발해 · 태봉 · 후백제는 명확하지 않다. 고려시대에는 1387년(우왕 13) 당시까지 통용된 원나라 제도에 의거한 품대와 호복을 혁파하고 명나라 제도에 의거하여 1∼9품까지 모두 사모단령을 착복하였다. 1품 중대광 이상은 꽃무늬를 새긴 금장식의 삽화금대, 2품 양부 이상은 조각하지 않은 민금으로 장식한 소금대, 개성윤 및 3품 대사헌으로부터 상시까지는 삽화은대, 판사부터 4품까지는 소은대, 5·6품에서 7품 이하 문하녹사 · 주서 · 밀직당후 · 삼사도사 · 예문춘추관과 전교시 및 성균관의 8·9품, 외방현령 · 감무는 뿔로 장식한 각대, 동서반 7품 이하는 비단으로 된 사대 등을 각각 착용하도록 개정되었다. 1391년(공양왕 3) 평양부 토관의 관복 상정에 수반되어 품대가 보완, 규정되었고, 이것이 고려 멸망 때까지 계속되었다. 조선에서는 1392년(태조 1) 고려 말의 제도를 계승하였다. 1품은 코뿔소의 뿔로 장식한 서대, 2품은 껍질이 단단하고 붉은 여지 같이 금색 바탕에 진홍점을 찍은 장식의 여지금대, 3품 이하는 흑색뿔로 장식한 흑각대를 착용하면서 비롯되었다. 1416년(태종 16) 1·2품은 

In [ ]:
# import pandas as pd
# import re

# pd.set_option("display.max_colwidth", None)  # 내용 자르지 않음

# df = pd.read_csv("./backend/db_pipeline/data/encykorea_cleaned6.csv")

# keywords = ["세종"]
# pattern = "|".join(map(re.escape, keywords))

# mask = df["contents"].str.contains(pattern, na=False)

# # 조회할 카테고리 지정
# target_cat = "문헌"  # 예: "전쟁사"

# # 카테고리까지 필터링
# filtered = df.loc[mask & (df["category"] == target_cat), ["category", "contents"]]

# # 줄바꿈 살려서 출력
# for idx, row in filtered.iterrows():
#     print(f"[{idx}] category: {row['category']}\n{str(row['contents']).replace('\\n', '\n')}")
#     print("-" * 40)

# print("선택 카테고리 행 개수:", len(filtered))